# Модель

## Данные

Признаки собираются функцией `build_features` из `features.py`, модели и подбор гиперпараметров в `models.py`. Признаки считаются только по событиям внутри окна наблюдения. Разбиение по времени: train это окна с 06.04 по 15.04, test это окна с 16.04 по 19.04. Гиперпараметры подбираются только на train по out-of-fold P@R с фолдами по дате окна, test используется один раз для итоговой оценки каждой модели.

In [1]:
import pandas as pd
from IPython.display import display

from features import build_features, load_data

DATA_DIR = '../data'
MODEL_PATH = '../models/catboost.joblib'
PARAMS_PATH = '../models/best_params.json'
TEST_START = pd.Timestamp('2026-04-16')
DISPLAY_LIMIT = 100
pd.set_option('display.max_columns', DISPLAY_LIMIT)
pd.set_option('display.max_rows', DISPLAY_LIMIT)
cookies, events = load_data(DATA_DIR)
features = build_features(events, cookies)
cookie_meta = cookies.set_index('cookie_id')
is_test = cookie_meta.window_start_ts >= TEST_START
X_train = features.loc[cookie_meta.index[~is_test]]
X_test = features.loc[cookie_meta.index[is_test]]
Y_train = cookie_meta.target[~is_test]
Y_test = cookie_meta.target[is_test]
window_dates = {
    'train': cookie_meta.window_start_ts[~is_test],
    'test': cookie_meta.window_start_ts[is_test],
}
window_ranges = [
    f'{dates.min():%d.%m} - {dates.max():%d.%m}'
    for dates in window_dates.values()
]
split_summary = pd.DataFrame(
    {
        'кук': [len(X_train), len(X_test)],
        'ботов': [Y_train.sum(), Y_test.sum()],
        'доля ботов': [f'{Y_train.mean():.1%}', f'{Y_test.mean():.1%}'],
        'окна': window_ranges,
    },
    index=list(window_dates),
)
print('Разбиение по дате начала окна')
display(split_summary)
print(f'признаков: {X_train.shape[1]}')

Разбиение по дате начала окна


,кук,ботов,доля ботов,окна
train,8400,679,8.1%,06.04 - 15.04
test,2691,220,8.2%,16.04 - 19.04


признаков: 58


Пример признаков для нескольких кук обучающей выборки.

In [2]:
X_train.head()

,cookie_age_days,n_events,item_view_share,search_results_view_share,photo_swipe_share,favorite_add_share,seller_page_view_share,contact_phone_show_share,contact_chat_open_share,contact_message_sent_share,login_share,login_count,gap_median_seconds,gap_min_seconds,gap_mean_seconds,gap_std_seconds,gap_cv,max_gap_hours,night_event_share,zero_second_share,same_ts_share,active_hours,session_span_hours,hour_entropy,hour_entropy_normalized,n_user_agents,automation_event_share,platform_combination,browser_family,browser_version,os_name,os_version,ua_kind,n_unique_categories,n_unique_locations,main_category,n_item_events,n_unique_items,events_per_item,repeat_view_share,viewed_items_with_photo_swipe_share,pro_seller_share,contacts_per_item_view,contact_without_view_share,n_search_events,n_unique_queries,max_search_page,deep_page_share,next_page_share,searches_per_query,n_pointer_events,pointer_x_mean,pointer_x_std,pointer_y_std,screen_edge_share,round_coordinate_share,repeated_point_share,pointer_missing_share
cookie_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,135.603692,7,0.428571,0.428571,0.142857,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,21.0,1.0,27.833333,29.949402,1.076026,0.023333,1.00000,0.000000,0.000000,1,0.046389,-0.000000,-0.000000,1,0.0,android,native_app,120.0,android,13.0,native_app,3,5,kvartiry_prodazha,4.0,4.0,1.000000,0.000000,0.000000,0.750000,0.000000,NaN,3.0,3.0,6.0,0.333333,0.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ck_7e4de46eeab82974,194.576111,41,0.536585,0.195122,0.097561,0.024390,0.097561,0.024390,0.000000,0.000000,0.024390,1,57.5,2.0,986.900000,2416.286861,2.448360,2.364167,0.00000,0.000000,0.000000,8,10.965556,2.679173,0.584339,1,0.0,desktop + web,yandex_browser,8.0,windows,NaN,desktop_browser,1,12,kvartiry_arenda,32.0,19.0,1.684211,0.363636,0.142857,0.343750,0.045455,1.0,8.0,5.0,8.0,0.500000,0.0,1.6,38.0,917.868421,509.231969,272.953569,0.0,0.0,0.000000,0.073171
ck_9320229ef6304522,32.994421,36,0.194444,0.361111,0.083333,0.083333,0.083333,0.055556,0.027778,0.027778,0.083333,3,22.0,1.0,1042.600000,2306.449400,2.212209,2.319444,0.50000,0.027778,0.000000,7,10.136389,2.660072,0.580173,1,0.0,desktop + web,chrome,116.0,windows,NaN,desktop_browser,7,9,detskie_tovary,20.0,19.0,1.052632,0.142857,0.000000,0.550000,0.571429,1.0,13.0,10.0,10.0,0.307692,0.0,1.3,35.0,936.457143,602.124295,312.027426,0.0,0.0,0.000000,0.027778
ck_30ccd25bc1714ed9,0.546759,21,0.666667,0.142857,0.000000,0.000000,0.095238,0.047619,0.047619,0.000000,0.000000,0,54.5,6.0,861.050000,2502.081501,2.905849,2.390000,0.52381,0.000000,0.000000,3,4.783611,1.441167,0.328111,1,0.0,desktop + web,chrome,118.0,windows,NaN,desktop_browser,1,4,detskie_tovary,18.0,10.0,1.800000,0.357143,0.000000,0.388889,0.142857,1.0,3.0,3.0,2.0,0.000000,0.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,1.000000
ck_a77c5f05948cdeef,94.837095,29,0.448276,0.206897,0.137931,0.068966,0.103448,0.034483,0.000000,0.000000,0.000000,0,51.5,0.0,682.071429,1720.954514,2.523129,2.006667,0.00000,0.000000,0.034483,5,5.305000,1.510562,0.329460,1,0.0,desktop + web,chrome,114.0,macos,NaN,desktop_browser,4,6,kvartiry_prodazha,23.0,16.0,1.437500,0.076923,0.083333,0.304348,0.076923,1.0,6.0,6.0,7.0,0.333333,0.0,1.0,28.0,961.392857,450.787816,297.263925,0.0,0.0,0.035714,0.034483


Пропуски не заполняются на этапе признаков: у мобильных кук нет курсора, у части кук нет поиска или объявлений, и каждая модель обрабатывает это по-своему.

In [3]:
feature_overview = pd.DataFrame(
    {
        'тип': X_train.dtypes.astype(str),
        'заполнен в train': X_train.notna()
        .mean()
        .map(lambda share: f'{share:.0%}'),
        'заполнен в test': X_test.notna()
        .mean()
        .map(lambda share: f'{share:.0%}'),
    }
)
print('Все признаки: тип и доля кук, у которых признак посчитан')
display(feature_overview)

Все признаки: тип и доля кук, у которых признак посчитан


,тип,заполнен в train,заполнен в test
cookie_age_days,float64,100%,100%
n_events,int64,100%,100%
item_view_share,float64,100%,100%
search_results_view_share,float64,100%,100%
photo_swipe_share,float64,100%,100%
favorite_add_share,float64,100%,100%
seller_page_view_share,float64,100%,100%
contact_phone_show_share,float64,100%,100%
contact_chat_open_share,float64,100%,100%
contact_message_sent_share,float64,100%,100%


## Бейзлайн на правилах

Пороги и веса правил подбираются на train по P@R при recall 0.75, а не 0.7. При оптимизации ровно на 0.7 Optuna подгоняет пороги так, что на train правила ловят 70% ботов с минимальным запасом (P@R 0.275), а на test с теми же параметрами recall 0.7 уже не достигается. Тогда единственный допустимый порог это отметить всех, и метрика падает до доли ботов (0.082). Запас по recall делает пороги устойчивее. Значение на train в выводе ниже посчитано при recall 0.75, на test при целевом 0.7.

In [4]:
from metric import precision_at_recall
from models import rule_score, tune_rules

study_rules = tune_rules(X_train, Y_train)
rule_test_score = rule_score(X_test, study_rules.best_params)
print(f'P@R>=0.7 на train: {study_rules.best_value:.3f}')
print(f'P@R>=0.7 на test: {precision_at_recall(Y_test, rule_test_score):.3f}')
print(study_rules.best_params)

P@R>=0.7 на train: 0.231
P@R>=0.7 на test: 0.227
{'fast_gap_seconds': 27, 'slow_gap_seconds': 284, 'pointer_x_std': 400, 'n_unique_locations': 19, 'max_search_page': 23, 'searches_per_query': 5.3735247066582685, 'weight_0': 0.9662034168553779, 'weight_1': 0.7728767035110873, 'weight_2': 0.05969975783615521, 'weight_3': 0.03250480882173533, 'weight_4': 0.5176726103864011, 'weight_5': 0.24249391018670394}


## Логистическая регрессия

Линейный бейзлайн: числа заполняются медианой с флагом пропуска и стандартизуются, категории кодируются OHE, `C` подбирается по out-of-fold P@R на train с фолдами по дате окна.

In [5]:
from models import (
    bot_probability,
    build_logistic_regression,
    tune_logistic_regression,
)

train_window_days = cookie_meta.window_start_ts.loc[X_train.index].dt.date
study_logistic = tune_logistic_regression(X_train, Y_train, train_window_days)
logistic_regression = build_logistic_regression(
    study_logistic.best_params['C']
)
logistic_regression.fit(X_train, Y_train)
logistic_test_score = bot_probability(logistic_regression, X_test)
print(f'лучший C: {study_logistic.best_params["C"]:.4f}')
print(f'P@R>=0.7 на train (out-of-fold): {study_logistic.best_value:.3f}')
print(
    f'P@R>=0.7 на test: {precision_at_recall(Y_test, logistic_test_score):.3f}'
)

лучший C: 0.0137
P@R>=0.7 на train (out-of-fold): 0.548
P@R>=0.7 на test: 0.471


## CatBoost

CatBoost сам обрабатывает пропуски в числовых признаках, балансировку классов (`auto_class_weights`) выбирает Optuna, а пропуски в категориях заменяются отдельным значением `missing` (`prepare_for_catboost` внутри пайплайна из `build_catboost`, обучение через `fit_catboost`).

In [6]:
from models import fit_catboost, tune_catboost

study_catboost = tune_catboost(X_train, Y_train, train_window_days)
catboost = fit_catboost(study_catboost.best_params, X_train, Y_train)
catboost_test_score = bot_probability(catboost, X_test)
print(f'P@R>=0.7 на train (out-of-fold): {study_catboost.best_value:.3f}')
print(
    f'P@R>=0.7 на test: {precision_at_recall(Y_test, catboost_test_score):.3f}'
)
print(study_catboost.best_params)

P@R>=0.7 на train (out-of-fold): 0.800
P@R>=0.7 на test: 0.737
{'iterations': 496, 'learning_rate': 0.10138509680176677, 'depth': 5, 'l2_leaf_reg': 10.370946433705793, 'random_strength': 0.536020094087912, 'bagging_temperature': 0.1851759224863634, 'auto_class_weights': None}


## Итог

Итоговая модель это CatBoost с подобранными параметрами. Ниже P@R на test для всех моделей, доля ботов в test соответствует случайному скору. CatBoost, обученный на train, сохраняется в `models/catboost.joblib`, подобранные параметры всех моделей в `models/best_params.json`.

In [9]:
import joblib

from models import save_params

best_params = {
    'rules': study_rules.best_params,
    'logistic_regression': study_logistic.best_params,
    'catboost': study_catboost.best_params,
}
save_params(best_params, PARAMS_PATH)
joblib.dump(catboost, MODEL_PATH)
test_scores = {
    'правила с порогами от Optuna': rule_test_score,
    'логистическая регрессия': logistic_test_score,
    'CatBoost, подобранные параметры': catboost_test_score,
}
test_precision = {
    name: precision_at_recall(Y_test, score)
    for name, score in test_scores.items()
}
test_results = pd.Series(
    {'случайный скор (доля ботов)': Y_test.mean(), **test_precision}
)
test_table = test_results.map(lambda precision: f'{precision:.3f}')
print('P@R>=0.7 на test по моделям')
print(test_table.to_string())

P@R>=0.7 на test по моделям
случайный скор (доля ботов)        0.082
правила с порогами от Optuna       0.227
логистическая регрессия            0.471
CatBoost, подобранные параметры    0.737
